# Lab 25 - Unified multicenter model optimization with nested LOCO

Compare Logistic Regression, CatBoost, LightGBM, and a small MLP under one locked protocol.

- Outer LOCO: each hospital is held out once as test.
- Inner 3-fold GroupKFold: tuning uses only the other three hospitals.
- Preprocessing is fitted inside every training split.
- Positive-class weighting is tuned on inner training data only.
- Stacking uses inner OOF predictions and is enabled only if both worst-site OOF ROC-AUC and Recall improve by at least 0.005.
- Outer test metrics are reported for comparison and are never used by the stacking gate.

In [ ]:
!pip -q install optuna lightgbm catboost seaborn

import json
import random
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import tensorflow as tf
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score,
    brier_score_loss, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
RANDOM_STATE = 42
N_TRIALS = 20
MAX_EPOCHS = 150
PATIENCE = 15
THRESHOLD = 0.50
EVAL_SEEDS = (42, 123, 2025)
STACKING_MIN_IMPROVEMENT = 0.005
STACKING_OOF_TRIALS = 10
MODEL_NAMES = ['Logistic Regression', 'CatBoost', 'LightGBM', 'MLP']
OUTPUT_DIR = Path('/content/uci_multicenter_models_loco_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach',
            'exang','oldpeak','slope','ca','thal']
TARGET = 'target'
NUMERICAL_FEATURES = ['age','trestbps','chol','thalach','oldpeak']
CATEGORICAL_FEATURES = ['sex','cp','fbs','restecg','exang','slope','ca','thal']
BASE_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease'
FILES = {'cleveland':'processed.cleveland.data', 'hungarian':'processed.hungarian.data',
         'switzerland':'processed.switzerland.data', 'va':'processed.va.data'}
SITES = list(FILES.keys())
COLUMNS = FEATURES + ['num']
LOCAL_DATA_DIR_CANDIDATES = [
    Path('/content/heart-disease-diagnosis/data/raw/uci_multicenter'),
    Path('/content/data/raw/uci_multicenter'),
    Path('data/raw/uci_multicenter'),
    Path('../data/raw/uci_multicenter'),
]
LOCAL_DATA_DIR = next((path for path in LOCAL_DATA_DIR_CANDIDATES if path.exists()), None)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

def read_uci(site, filename):
    source = (LOCAL_DATA_DIR / filename) if LOCAL_DATA_DIR else f'{BASE_URL}/{filename}'
    frame = pd.read_csv(source, names=COLUMNS, na_values=['?'], skipinitialspace=True)
    frame = frame.apply(pd.to_numeric, errors='coerce')
    frame[TARGET] = (frame['num'] > 0).astype('int8')
    frame['site'] = site
    return frame[FEATURES + [TARGET, 'site']]

data = pd.concat([read_uci(site, filename) for site, filename in FILES.items()], ignore_index=True)
assert len(data) == 920, f'Expected 920 rows, got {len(data)}'
display(data.groupby('site')[TARGET].agg(['size','sum','mean']).round(4))
source_label = str(LOCAL_DATA_DIR) if LOCAL_DATA_DIR else 'UCI URL fallback'
print('TensorFlow:', tf.__version__, '| trials:', N_TRIALS, '| seeds:', EVAL_SEEDS)
print('Data source:', source_label)

In [ ]:
FIXED_PARAMS = {
    'Logistic Regression': {'C': 1.0, 'solver': 'lbfgs', 'max_iter': 2000, 'positive_weight': 1.0},
    'CatBoost': {'iterations': 300, 'depth': 5, 'learning_rate': 0.03, 'l2_leaf_reg': 3.0,
                 'random_strength': 1.0, 'border_count': 64, 'positive_weight': 1.0},
    'LightGBM': {'n_estimators': 250, 'learning_rate': 0.03, 'num_leaves': 15, 'max_depth': -1,
                 'min_child_samples': 15, 'subsample': 0.9, 'colsample_bytree': 0.9,
                 'reg_lambda': 1.0, 'positive_weight': 1.0},
    'MLP': {'hidden1': 32, 'hidden2': 16, 'dropout': 0.20, 'learning_rate': 0.001,
            'l2_reg': 1e-4, 'batch_size': 32, 'positive_weight': 1.0,
            'max_epochs': MAX_EPOCHS, 'patience': PATIENCE},
}

def apply_p1(frame):
    out = frame.copy()
    for column in FEATURES:
        out[column] = pd.to_numeric(out[column], errors='coerce')
    for column in ['trestbps', 'chol']:
        out.loc[out[column] <= 0, column] = np.nan
    return out

def prepare_dense(frame):
    ready = apply_p1(frame)
    return ready[FEATURES], ready[TARGET].to_numpy()

def prepare_catboost(frame):
    ready = apply_p1(frame)
    features = ready[FEATURES].copy()
    for column in FEATURES:
        features[f'{column}__missing'] = features[column].isna().astype('int8')
    for column in CATEGORICAL_FEATURES:
        features[column] = features[column].fillna('__MISSING__').astype(str)
    return features, ready[TARGET].to_numpy()

def make_preprocessor(scale_numeric=True):
    numeric_steps = [('imputer', SimpleImputer(strategy='median', add_indicator=True))]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler()))
    return ColumnTransformer([
        ('numeric', Pipeline(numeric_steps), NUMERICAL_FEATURES),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), CATEGORICAL_FEATURES),
    ])

def make_preprocessor_compat(scale_numeric=True):
    try:
        return make_preprocessor(scale_numeric)
    except TypeError:
        numeric_steps = [('imputer', SimpleImputer(strategy='median', add_indicator=True))]
        if scale_numeric:
            numeric_steps.append(('scaler', StandardScaler()))
        return ColumnTransformer([
            ('numeric', Pipeline(numeric_steps), NUMERICAL_FEATURES),
            ('categorical', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
                ('encoder', OneHotEncoder(handle_unknown='ignore', sparse=False)),
            ]), CATEGORICAL_FEATURES),
        ])

def build_estimator(model_name, params, seed):
    params = dict(params)
    positive_weight = float(params.pop('positive_weight', 1.0))
    if model_name == 'Logistic Regression':
        return LogisticRegression(random_state=seed, class_weight={0: 1.0, 1: positive_weight}, **params)
    if model_name == 'CatBoost':
        return CatBoostClassifier(loss_function='Logloss', eval_metric='AUC', random_seed=seed,
            scale_pos_weight=positive_weight, verbose=False, allow_writing_files=False,
            thread_count=2, **params)
    if model_name == 'LightGBM':
        return LGBMClassifier(random_state=seed, verbosity=-1, n_jobs=1,
            scale_pos_weight=positive_weight, **params)
    raise ValueError(f'Unknown estimator: {model_name}')

def build_mlp(input_dim, params, seed):
    seed_everything(seed)
    model = Sequential([Input(shape=(input_dim,)),
        Dense(int(params['hidden1']), activation='relu', kernel_regularizer=l2(float(params['l2_reg']))),
        Dropout(float(params['dropout'])),
        Dense(int(params['hidden2']), activation='relu', kernel_regularizer=l2(float(params['l2_reg']))),
        Dropout(float(params['dropout'])), Dense(1, activation='sigmoid')])
    model.compile(optimizer=Adam(learning_rate=float(params['learning_rate'])),
                  loss='binary_crossentropy', metrics=[tf.keras.metrics.AUC(name='auc')])
    return model

def fit_mlp(X_train, y_train, X_valid, y_valid, params, seed):
    tf.keras.backend.clear_session()
    model = build_mlp(X_train.shape[1], params, seed)
    reduce_lr = ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5,
        patience=max(3, int(params['patience']) // 3), min_lr=1e-6, verbose=0)
    early_stop = EarlyStopping(monitor='val_auc', mode='max', patience=int(params['patience']),
        min_delta=1e-4, restore_best_weights=True, verbose=0)
    class_weight = {0: 1.0, 1: float(params.get('positive_weight', 1.0))}
    history = model.fit(X_train, y_train, validation_data=(X_valid, y_valid),
        epochs=int(params['max_epochs']), batch_size=int(params['batch_size']),
        callbacks=[reduce_lr, early_stop], class_weight=class_weight, verbose=0)
    values = history.history.get('val_auc', [])
    best_epoch = int(np.argmax(values) + 1) if values else len(history.history['loss'])
    return model, best_epoch

def fit_mlp_full(X_train, y_train, params, epochs, seed):
    tf.keras.backend.clear_session()
    model = build_mlp(X_train.shape[1], params, seed)
    class_weight = {0: 1.0, 1: float(params.get('positive_weight', 1.0))}
    model.fit(X_train, y_train, epochs=int(epochs), batch_size=int(params['batch_size']),
              class_weight=class_weight, verbose=0, shuffle=True)
    return model

def fit_predict_base(model_name, train_frame, predict_frame, params, seed, validation_frame=None):
    params = dict(params)
    if model_name in ['Logistic Regression', 'LightGBM']:
        X_train, y_train = prepare_dense(train_frame)
        X_predict, _ = prepare_dense(predict_frame)
        preprocessor = make_preprocessor_compat(scale_numeric=(model_name == 'Logistic Regression'))
        X_train = preprocessor.fit_transform(X_train).astype('float32')
        X_predict = preprocessor.transform(X_predict).astype('float32')
        estimator = build_estimator(model_name, params, seed)
        estimator.fit(X_train, y_train)
        return estimator.predict_proba(X_predict)[:, 1]
    if model_name == 'CatBoost':
        X_train, y_train = prepare_catboost(train_frame)
        X_predict, _ = prepare_catboost(predict_frame)
        estimator = build_estimator(model_name, params, seed)
        estimator.fit(X_train, y_train, cat_features=CATEGORICAL_FEATURES)
        return estimator.predict_proba(X_predict)[:, 1]
    if model_name == 'MLP':
        X_train, y_train = prepare_dense(train_frame)
        X_predict, _ = prepare_dense(predict_frame)
        preprocessor = make_preprocessor_compat(scale_numeric=True)
        X_train = preprocessor.fit_transform(X_train).astype('float32')
        X_predict = preprocessor.transform(X_predict).astype('float32')
        if validation_frame is not None:
            X_valid, y_valid = prepare_dense(validation_frame)
            X_valid = preprocessor.transform(X_valid).astype('float32')
            model, _ = fit_mlp(X_train, y_train, X_valid, y_valid, params, seed)
        else:
            epochs = int(params.get('fit_epochs', params.get('max_epochs', MAX_EPOCHS)))
            model = fit_mlp_full(X_train, y_train, params, epochs, seed)
        return model.predict(X_predict, batch_size=256, verbose=0).ravel()
    raise ValueError(f'Unknown model: {model_name}')

def score_probability(y_true, probability):
    prediction = (probability >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {'accuracy': accuracy_score(y_true, prediction),
        'precision': precision_score(y_true, prediction, zero_division=0),
        'recall': recall_score(y_true, prediction, zero_division=0),
        'pr_auc': average_precision_score(y_true, probability),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'f1': f1_score(y_true, prediction, zero_division=0),
        'roc_auc': roc_auc_score(y_true, probability) if len(np.unique(y_true)) > 1 else np.nan,
        'brier': brier_score_loss(y_true, probability),
        'false_negatives': int(fn), 'false_positives': int(fp)}

In [ ]:
def suggest_params(trial, model_name):
    if model_name == 'Logistic Regression':
        return {'C': trial.suggest_float('C', 0.01, 10.0, log=True),
                'solver': trial.suggest_categorical('solver', ['lbfgs', 'liblinear']),
                'max_iter': 2000,
                'positive_weight': trial.suggest_categorical('positive_weight', [0.75, 1.0, 1.25, 1.5, 2.0])}
    if model_name == 'CatBoost':
        return {'iterations': trial.suggest_int('iterations', 150, 600),
                'depth': trial.suggest_int('depth', 3, 7),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.20, log=True),
                'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 20.0, log=True),
                'random_strength': trial.suggest_float('random_strength', 0.0, 3.0),
                'border_count': trial.suggest_categorical('border_count', [32, 64, 128]),
                'positive_weight': trial.suggest_categorical('positive_weight', [0.75, 1.0, 1.25, 1.5, 2.0])}
    if model_name == 'LightGBM':
        return {'n_estimators': trial.suggest_int('n_estimators', 100, 500),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.20, log=True),
                'num_leaves': trial.suggest_int('num_leaves', 7, 63),
                'max_depth': trial.suggest_categorical('max_depth', [-1, 3, 5, 8]),
                'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
                'subsample': trial.suggest_float('subsample', 0.70, 1.00),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.70, 1.00),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
                'positive_weight': trial.suggest_categorical('positive_weight', [0.75, 1.0, 1.25, 1.5, 2.0])}
    if model_name == 'MLP':
        return {'hidden1': trial.suggest_categorical('hidden1', [16, 32, 64]),
                'hidden2': trial.suggest_categorical('hidden2', [8, 16, 32]),
                'dropout': trial.suggest_float('dropout', 0.0, 0.40),
                'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True),
                'l2_reg': trial.suggest_float('l2_reg', 1e-6, 1e-2, log=True),
                'batch_size': trial.suggest_categorical('batch_size', [16, 32, 64]),
                'positive_weight': trial.suggest_categorical('positive_weight', [0.75, 1.0, 1.25, 1.5, 2.0]),
                'max_epochs': MAX_EPOCHS, 'patience': PATIENCE}
    raise ValueError(f'Unknown model: {model_name}')

def tune_model_on_training(train_frame, model_name, seed, n_splits=3, n_trials=N_TRIALS):
    y = train_frame[TARGET].to_numpy()
    groups = train_frame['site'].to_numpy()
    splits = list(GroupKFold(n_splits=n_splits).split(train_frame, y, groups))

    def objective(trial):
        params = suggest_params(trial, model_name)
        fold_scores = []
        for fold_number, (fit_idx, valid_idx) in enumerate(splits):
            fit_frame = train_frame.iloc[fit_idx].reset_index(drop=True)
            valid_frame = train_frame.iloc[valid_idx].reset_index(drop=True)
            probability = fit_predict_base(model_name, fit_frame, valid_frame, params,
                                            seed + fold_number, validation_frame=valid_frame)
            fold_score = roc_auc_score(valid_frame[TARGET], probability)
            fold_scores.append(fold_score)
            trial.report(float(np.mean(fold_scores)), step=fold_number)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(fold_scores))

    study = optuna.create_study(direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=seed),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3))
    started = time.perf_counter()
    study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=False)
    return study, time.perf_counter() - started

In [ ]:
def select_mlp_epochs(train_frame, params, seed, n_splits=3):
    y = train_frame[TARGET].to_numpy()
    groups = train_frame['site'].to_numpy()
    epoch_values = []
    for fold_number, (fit_idx, valid_idx) in enumerate(
        GroupKFold(n_splits=n_splits).split(train_frame, y, groups)):
        fit_frame = train_frame.iloc[fit_idx].reset_index(drop=True)
        valid_frame = train_frame.iloc[valid_idx].reset_index(drop=True)
        X_fit, y_fit = prepare_dense(fit_frame)
        X_valid, y_valid = prepare_dense(valid_frame)
        preprocessor = make_preprocessor_compat(scale_numeric=True)
        X_fit = preprocessor.fit_transform(X_fit).astype('float32')
        X_valid = preprocessor.transform(X_valid).astype('float32')
        _, best_epoch = fit_mlp(X_fit, y_fit, X_valid, y_valid, params, seed + fold_number)
        epoch_values.append(best_epoch)
    return max(1, int(np.median(epoch_values))), epoch_values

def make_nested_oof_predictions(train_frame, seed):
    # Each OOF hospital is predicted by models tuned without that hospital.
    y = train_frame[TARGET].reset_index(drop=True).to_numpy()
    groups = train_frame['site'].reset_index(drop=True).to_numpy()
    oof = pd.DataFrame(index=np.arange(len(train_frame)), columns=MODEL_NAMES, dtype='float64')
    outer_splits = list(GroupKFold(n_splits=3).split(train_frame, y, groups))
    for oof_fold, (fit_idx, valid_idx) in enumerate(outer_splits):
        fit_frame = train_frame.iloc[fit_idx].reset_index(drop=True)
        valid_frame = train_frame.iloc[valid_idx].reset_index(drop=True)
        fold_params = {}
        fold_epochs = {}
        for model_number, model_name in enumerate(MODEL_NAMES):
            fold_seed = seed + oof_fold * 100 + model_number
            study, _ = tune_model_on_training(
                fit_frame, model_name, fold_seed, n_splits=2, n_trials=STACKING_OOF_TRIALS)
            params = {**FIXED_PARAMS[model_name], **study.best_trial.params}
            if model_name == 'MLP':
                fold_epochs[model_name], _ = select_mlp_epochs(
                    fit_frame, params, fold_seed + 50, n_splits=2)
                params['fit_epochs'] = fold_epochs[model_name]
            fold_params[model_name] = params
        for model_number, model_name in enumerate(MODEL_NAMES):
            probability = fit_predict_base(
                model_name, fit_frame, valid_frame, fold_params[model_name],
                seed + oof_fold * 100 + model_number)
            oof.loc[valid_idx, model_name] = probability
    assert not oof.isna().any().any(), 'OOF predictions are incomplete'
    meta_model = LogisticRegression(C=1.0, max_iter=2000)
    meta_model.fit(oof[MODEL_NAMES], y)
    oof['Stacking'] = meta_model.predict_proba(oof[MODEL_NAMES])[:, 1]

    site_rows = []
    site_labels = train_frame['site'].reset_index(drop=True)
    for model_name in MODEL_NAMES + ['Stacking']:
        for site in sorted(site_labels.unique()):
            mask = site_labels == site
            site_rows.append({'model': model_name, 'site': site,
                              **score_probability(y[mask], oof.loc[mask, model_name].to_numpy())})
    site_metrics = pd.DataFrame(site_rows)
    worst_metrics = site_metrics.groupby('model').agg(
        roc_auc_worst=('roc_auc', 'min'), recall_worst=('recall', 'min'),
        pr_auc_worst=('pr_auc', 'min'), brier_worst=('brier', 'max')).reset_index()
    base_reference = (worst_metrics[worst_metrics['model'].isin(MODEL_NAMES)]
                     .sort_values(['roc_auc_worst', 'recall_worst'], ascending=False).iloc[0])
    stack_metrics = worst_metrics[worst_metrics['model'] == 'Stacking'].iloc[0]
    enabled = (stack_metrics['roc_auc_worst'] >= base_reference['roc_auc_worst'] + STACKING_MIN_IMPROVEMENT
               and stack_metrics['recall_worst'] >= base_reference['recall_worst'] + STACKING_MIN_IMPROVEMENT)
    gate = {'enabled': bool(enabled), 'reference_model': base_reference['model'],
            'reference_oof_worst_auc': float(base_reference['roc_auc_worst']),
            'reference_oof_worst_recall': float(base_reference['recall_worst']),
            'stacking_oof_worst_auc': float(stack_metrics['roc_auc_worst']),
            'stacking_oof_worst_recall': float(stack_metrics['recall_worst']),
            'min_required_improvement': STACKING_MIN_IMPROVEMENT}
    return oof, meta_model, site_metrics, worst_metrics, gate

In [ ]:
outer_records = []
best_param_records = []
trial_records = []
stacking_gate_records = []
stacking_oof_site_records = []

for test_site in SITES:
    train_frame = data[data['site'] != test_site].reset_index(drop=True)
    test_frame = data[data['site'] == test_site].reset_index(drop=True)
    best_params_by_model = {}
    tuning_seconds_by_model = {}
    best_inner_auc_by_model = {}
    fixed_epochs_by_model = {}
    tuned_epochs_by_model = {}

    for model_name in MODEL_NAMES:
        if model_name == 'MLP':
            fixed_epochs_by_model[model_name], _ = select_mlp_epochs(
                train_frame, FIXED_PARAMS[model_name], RANDOM_STATE + 1)
        study, tuning_seconds = tune_model_on_training(train_frame, model_name, RANDOM_STATE)
        best_params = {**FIXED_PARAMS[model_name], **study.best_trial.params}
        best_params_by_model[model_name] = best_params
        tuning_seconds_by_model[model_name] = tuning_seconds
        best_inner_auc_by_model[model_name] = study.best_value
        if model_name == 'MLP':
            tuned_epochs_by_model[model_name], _ = select_mlp_epochs(
                train_frame, best_params, RANDOM_STATE + 2)
        best_param_records.append({
            'test_site': test_site, 'model': model_name, 'best_inner_roc_auc': study.best_value,
            'best_epoch': tuned_epochs_by_model.get(model_name, np.nan),
            'best_params': json.dumps(best_params, sort_keys=True),
            'tuning_seconds': tuning_seconds, 'n_trials': len(study.trials)})
        for trial in study.trials:
            trial_records.append({'test_site': test_site, 'model': model_name,
                'trial_number': trial.number, 'state': str(trial.state),
                'value': trial.value, 'params': json.dumps(trial.params, sort_keys=True)})

    tuned_test_probabilities = {seed: {} for seed in EVAL_SEEDS}
    for model_name in MODEL_NAMES:
        for eval_seed in EVAL_SEEDS:
            fixed_params = dict(FIXED_PARAMS[model_name])
            tuned_params = dict(best_params_by_model[model_name])
            if model_name == 'MLP':
                fixed_params['fit_epochs'] = fixed_epochs_by_model[model_name]
                tuned_params['fit_epochs'] = tuned_epochs_by_model[model_name]
            started = time.perf_counter()
            fixed_probability = fit_predict_base(model_name, train_frame, test_frame, fixed_params, eval_seed)
            fixed_fit_seconds = time.perf_counter() - started
            outer_records.append({'validation': 'LOCO', 'test_site': test_site, 'seed': eval_seed,
                'configuration': 'F0_P1_fixed', 'model': model_name, 'tuning_seconds': 0.0,
                'fit_seconds': fixed_fit_seconds, 'best_inner_roc_auc': np.nan,
                'best_params': json.dumps(FIXED_PARAMS[model_name], sort_keys=True),
                **score_probability(test_frame[TARGET].to_numpy(), fixed_probability)})

            started = time.perf_counter()
            tuned_probability = fit_predict_base(model_name, train_frame, test_frame, tuned_params, eval_seed)
            tuned_fit_seconds = time.perf_counter() - started
            tuned_test_probabilities[eval_seed][model_name] = tuned_probability
            outer_records.append({'validation': 'LOCO', 'test_site': test_site, 'seed': eval_seed,
                'configuration': 'F0_P1_optuna_nested', 'model': model_name,
                'tuning_seconds': tuning_seconds_by_model[model_name], 'fit_seconds': tuned_fit_seconds,
                'best_inner_roc_auc': best_inner_auc_by_model[model_name],
                'best_params': json.dumps(best_params_by_model[model_name], sort_keys=True),
                **score_probability(test_frame[TARGET].to_numpy(), tuned_probability)})

    oof, meta_model, oof_site_metrics, oof_worst_metrics, gate = make_nested_oof_predictions(
        train_frame, RANDOM_STATE + 10)
    gate['outer_test_site'] = test_site
    stacking_gate_records.append(gate)
    oof_site_copy = oof_site_metrics.copy()
    oof_site_copy['outer_test_site'] = test_site
    stacking_oof_site_records.extend(oof_site_copy.to_dict('records'))

    if gate['enabled']:
        for eval_seed in EVAL_SEEDS:
            stacked_features = np.column_stack([tuned_test_probabilities[eval_seed][model_name]
                for model_name in MODEL_NAMES])
            stacking_probability = meta_model.predict_proba(stacked_features)[:, 1]
            outer_records.append({'validation': 'LOCO', 'test_site': test_site, 'seed': eval_seed,
                'configuration': 'stacking_oof_gated', 'model': 'Stacking',
                'tuning_seconds': np.nan, 'fit_seconds': np.nan, 'best_inner_roc_auc': np.nan,
                'best_params': json.dumps({'base_models': MODEL_NAMES, 'gate': gate}, sort_keys=True),
                **score_probability(test_frame[TARGET].to_numpy(), stacking_probability)})
    print('Completed outer test site:', test_site, '| stacking enabled:', gate['enabled'])

results_df = pd.DataFrame(outer_records)
best_params_df = pd.DataFrame(best_param_records)
trials_df = pd.DataFrame(trial_records)
stacking_gate_df = pd.DataFrame(stacking_gate_records)
stacking_oof_site_df = pd.DataFrame(stacking_oof_site_records)
display(results_df.round(4))
display(best_params_df)
display(stacking_gate_df)

In [ ]:
summary = results_df.groupby(['configuration', 'model']).agg(
    eval_rows=('seed', 'size'), folds=('test_site', 'nunique'), seeds=('seed', 'nunique'),
    roc_auc_mean=('roc_auc', 'mean'), roc_auc_std=('roc_auc', 'std'),
    pr_auc_mean=('pr_auc', 'mean'), recall_mean=('recall', 'mean'),
    recall_std=('recall', 'std'), specificity_mean=('specificity', 'mean'),
    f1_mean=('f1', 'mean'), brier_mean=('brier', 'mean'),
    false_negatives_mean_per_site_seed=('false_negatives', 'mean'),
    false_positives_mean_per_site_seed=('false_positives', 'mean'),
    best_inner_roc_auc_mean=('best_inner_roc_auc', 'mean'),
    tuning_seconds_mean=('tuning_seconds', 'mean'), fit_seconds_mean=('fit_seconds', 'mean')).reset_index()

site_summary = results_df.groupby(['configuration', 'model', 'test_site']).agg(
    roc_auc=('roc_auc', 'mean'), pr_auc=('pr_auc', 'mean'), recall=('recall', 'mean'),
    specificity=('specificity', 'mean'), f1=('f1', 'mean'), brier=('brier', 'mean')).reset_index()
worst_site = site_summary.groupby(['configuration', 'model']).agg(
    roc_auc_worst=('roc_auc', 'min'), pr_auc_worst=('pr_auc', 'min'),
    recall_worst=('recall', 'min'), brier_worst=('brier', 'max')).reset_index()
seed_summary = results_df.groupby(['configuration', 'model', 'seed']).agg(
    roc_auc_mean=('roc_auc', 'mean'), pr_auc_mean=('pr_auc', 'mean'),
    recall_mean=('recall', 'mean'), brier_mean=('brier', 'mean')).reset_index()
seed_variance = seed_summary.groupby(['configuration', 'model']).agg(
    roc_auc_seed_std=('roc_auc_mean', 'std'), pr_auc_seed_std=('pr_auc_mean', 'std'),
    recall_seed_std=('recall_mean', 'std'), brier_seed_std=('brier_mean', 'std')).reset_index()
summary = summary.merge(worst_site, on=['configuration', 'model']).merge(
    seed_variance, on=['configuration', 'model'])

display(summary.sort_values(['roc_auc_worst', 'recall_worst'], ascending=False).round(6))
display(site_summary.sort_values(['configuration', 'roc_auc']))
display(seed_summary)
display(stacking_gate_df)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=summary, x='model', y='roc_auc_worst', hue='configuration', ax=axes[0])
axes[0].set_title('Worst-site ROC-AUC across outer LOCO')
axes[0].tick_params(axis='x', rotation=20)
sns.barplot(data=summary, x='model', y='recall_mean', hue='configuration', ax=axes[1])
axes[1].set_title('Mean Recall at threshold 0.50')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'model_comparison_summary.png', dpi=180, bbox_inches='tight')
plt.show()

## Stacking decision

For each outer fold, stacking is evaluated on inner OOF predictions only. It is enabled only when both worst-site OOF ROC-AUC and worst-site OOF Recall improve over the strongest base model by at least 0.005. If either condition fails, no stacking prediction is made for that outer test hospital.

In [ ]:
results_df.to_csv(OUTPUT_DIR / 'model_loco_results.csv', index=False)
summary.to_csv(OUTPUT_DIR / 'model_loco_summary.csv', index=False)
site_summary.to_csv(OUTPUT_DIR / 'model_loco_site_summary.csv', index=False)
seed_summary.to_csv(OUTPUT_DIR / 'model_loco_seed_summary.csv', index=False)
seed_variance.to_csv(OUTPUT_DIR / 'model_loco_seed_variance.csv', index=False)
best_params_df.to_csv(OUTPUT_DIR / 'best_params_by_outer_fold.csv', index=False)
trials_df.to_csv(OUTPUT_DIR / 'optuna_trial_history.csv', index=False)
stacking_gate_df.to_csv(OUTPUT_DIR / 'stacking_oof_gate_by_outer_fold.csv', index=False)
stacking_oof_site_df.to_csv(OUTPUT_DIR / 'stacking_oof_site_metrics.csv', index=False)
run_config = {
    'dataset_rows': 920, 'data_source': source_label, 'models': MODEL_NAMES,
    'validation': 'outer LOCO + inner 3-fold GroupKFold by hospital',
    'preprocessing': {
        'LR': 'P1 + train-fold median/mode imputation + missing indicators + one-hot + scaling',
        'CatBoost': 'P1 + native numerical missing + categorical missing token + missing indicators',
        'LightGBM': 'P1 + train-fold median/mode imputation + missing indicators + one-hot',
        'MLP': 'P1 + train-fold median/mode imputation + missing indicators + one-hot + scaling'},
    'n_trials_per_model_per_outer_fold': N_TRIALS, 'optimization_metric': 'inner mean ROC-AUC',
    'stacking_oof_trials_per_fold': STACKING_OOF_TRIALS,
    'tuning_seed': RANDOM_STATE, 'evaluation_seeds': list(EVAL_SEEDS), 'threshold': THRESHOLD,
    'class_weight': 'positive_weight tuned in inner CV and applied to train rows only',
    'smotenc': 'not used',
    'stacking_gate': 'inner OOF worst-site AUC and Recall must both improve by at least 0.005',
    'outer_test_policy': 'held-out hospital is never used for tuning, epoch selection, stacking gate, or model selection'},
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
zip_path = shutil.make_archive('/content/uci_multicenter_models_loco_results', 'zip', OUTPUT_DIR)
print('Saved:', OUTPUT_DIR, zip_path)